# 实验5-2：多层卷积神经网络的实现（学生练习版）

本实验基于 `experiment5_1_convolution_activation_basics.ipynb` 中实现的 Padding、单通道卷积和 ReLU，将固定卷积操作逐步扩展为可训练的两层卷积神经网络，并在 8×8 手写数字数据集上完成 0～9 十分类。

实验目标：

1. 理解多层 CNN 如何逐层提取图像特征。
2. 理解单图像、单卷积核操作如何扩展到批量、多通道、多卷积核计算。
3. 明确 CNN 训练在基础卷积之外需要增加的功能。
4. 手写卷积反向传播、ReLU、最大池化和 Softmax 交叉熵。
5. 实现两层卷积网络的前向传播、反向传播与参数更新。
6. 可视化训练曲线、预测结果、混淆矩阵和两层特征图。

> 本实验不依赖 PyTorch 或 TensorFlow，网络核心计算由 NumPy 显式实现。

## 学生任务与完成顺序

本练习只需要补全第 5 节中的 `Conv2DManual`，其他数据处理、池化、损失函数、网络组装、训练和可视化代码均已提供。建议按照以下顺序完成：

1. **运行第 1～4 节**：理解 CNN 训练比基础卷积增加了哪些功能，并准备数据、Padding 和 ReLU。
2. **完成参数初始化**：在 `__init__` 中创建卷积核 `weight`、偏置 `bias` 和缓存 `cache`。
3. **完成前向传播**：在 `forward` 中实现 Padding、输出尺寸计算、卷积核滑动和多通道乘加。
4. **检查前向输出形状**：运行第 7.1 节。第一层和第二层卷积输出应分别为 `(4, 4, 8, 8)` 和 `(4, 8, 4, 4)`。
5. **完成反向传播**：在 `backward` 中计算输入梯度、卷积核梯度和偏置梯度。
6. **运行完整训练**：运行第 8 节及后续单元，确认损失下降、准确率上升，并生成可视化结果。

请按 TODO 编号依次实现。当前阶段未完成时，先不要跳到后面的训练步骤。

## 1. 从基础卷积操作到可训练 CNN

实验5-1实现的是一幅二维图像与一个固定卷积核之间的前向计算。该操作能够生成特征图，但还不能通过数据自动学习卷积核，也不能直接完成分类任务。

### 1.1 基础卷积需要进行的扩展

| 实验5-1中的基础操作 | 训练 CNN 时的扩展 |
|---|---|
| 输入一幅二维图像 `(H, W)` | 一次输入一批图像 `(N, C, H, W)` |
| 使用一个卷积核 | 使用多个可训练卷积核，生成多个输出通道 |
| 卷积核处理单通道 | 每个卷积核同时处理所有输入通道并累加结果 |
| 卷积核数值由人工指定 | 随机初始化卷积核，并通过训练不断更新 |
| 只计算前向特征图 | 保存中间结果，并实现卷积反向传播 |
| 只观察卷积结果 | 使用分类损失评价预测结果，并计算准确率 |

### 1.2 在卷积操作基础上还要增加的功能

一个完整的卷积神经网络训练过程还需要：

1. **批量与多通道计算**：同时处理多幅图像，并支持上一卷积层输出的多张特征图。
2. **可训练参数**：将卷积核和偏置保存为网络参数，采用合适方式进行初始化。
3. **激活函数**：在卷积之后使用 ReLU 引入非线性，并实现其反向传播。
4. **池化层**：缩小特征图尺寸、降低计算量，并保留显著局部特征。
5. **分类器**：将最后的特征图展平，通过全连接层得到各类别得分。
6. **损失函数**：使用 Softmax 交叉熵衡量预测结果与真实标签的差异。
7. **反向传播**：从损失出发，反向计算全连接层、池化层、激活函数和卷积层的梯度。
8. **参数更新**：使用梯度下降更新卷积核、卷积偏置、全连接权重和偏置。
9. **训练循环**：按批次重复前向传播、损失计算、反向传播和参数更新，并在测试集上评价模型。

### 1.3 本实验的网络结构

第一层卷积从原始图像中提取边缘、方向等低层特征，第二层卷积将这些低层特征组合成更复杂的局部形状。

```text
输入图像 (1×8×8)
    ↓
Conv1：4 个 3×3 卷积核，padding=1
    ↓
ReLU1
    ↓
2×2 MaxPool1                     → (4×4×4)
    ↓
Conv2：8 个 3×3 卷积核，padding=1
    ↓
ReLU2
    ↓
2×2 MaxPool2                     → (8×2×2)
    ↓
展平为 32 维向量
    ↓
全连接层 + Softmax               → 10 类
```

两次池化将空间尺寸从 8×8 依次减小为 4×4 和 2×2，卷积通道数则从 1 增加为 4 和 8。

## 2. 实验准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("NumPy version:", np.__version__)

## 3. 读取手写数字数据

`load_digits` 包含 1797 幅 8×8 灰度图像。程序将像素从 0～16 归一化到 0～1，并按照 8:2 划分训练集和测试集。

In [ ]:
digits = load_digits()
images = digits.images.astype(np.float64) / 16.0
labels = digits.target.astype(np.int64)

# CNN 输入格式：(样本数, 通道数, 高, 宽)
images = images[:, np.newaxis, :, :]

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=labels,
)

print("train:", X_train.shape, y_train.shape)
print("test:", X_test.shape, y_test.shape)
print("pixel range:", X_train.min(), "～", X_train.max())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for digit, ax in enumerate(axes.ravel()):
    sample_index = np.where(y_train == digit)[0][0]
    ax.imshow(X_train[sample_index, 0], cmap="gray")
    ax.set_title(f"label: {digit}")
    ax.axis("off")

plt.suptitle("handwritten digit examples")
plt.tight_layout()
plt.show()

## 4. 复用并扩展实验5-1的基础操作

实验5-1的 Padding 处理二维图像。本实验将其扩展到 CNN 常用的 `(批量, 通道, 高, 宽)` 四维输入，同时继续使用手写 ReLU。

训练还需要 ReLU 的反向传播：正输入位置允许梯度通过，非正输入位置的梯度为 0。

In [ ]:
def zero_padding_batch_manual(x, padding=0):
    """将实验5-1的二维 Padding 扩展到 (N, C, H, W) 批量输入。"""
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 4:
        raise ValueError("x 必须采用 (N, C, H, W) 四维格式")
    if not isinstance(padding, int) or padding < 0:
        raise ValueError("padding 必须是非负整数")
    if padding == 0:
        return x.copy()

    batch_size, channels, height, width = x.shape
    padded = np.zeros(
        (batch_size, channels, height + 2 * padding, width + 2 * padding),
        dtype=x.dtype,
    )
    padded[:, :, padding:padding + height, padding:padding + width] = x
    return padded


def relu_manual(x):
    """ReLU 前向传播。"""
    x = np.asarray(x, dtype=np.float64)
    return np.where(x > 0, x, 0.0)


def relu_backward_manual(grad_output, x):
    """ReLU 反向传播。"""
    return grad_output * (x > 0)

## 5. 将单通道卷积改进为可训练卷积层

这里保留实验5-1中的核心计算方式：Padding、卷积核滑动、截取局部区域以及逐元素乘加。在此基础上增加批量维度、输入通道、输出通道、可训练权重、偏置和反向传播。

卷积核权重形状为：

```text
(输出通道数, 输入通道数, 卷积核高度, 卷积核宽度)
```

前向传播的每个输出值仍然来自局部区域与卷积核的乘加，只是需要对所有输入通道求和，并对每个输出卷积核分别计算。

反向传播新增三类梯度：

- `grad_weight`：用于更新卷积核。
- `grad_bias`：用于更新卷积偏置。
- `grad_x`：传递给前一层，使更早的卷积层也能得到梯度。

### `Conv2DManual` 补全步骤

#### 阶段一：初始化可训练参数

1. 使用随机数生成器和 He 初始化比例创建卷积核。
2. 卷积核形状必须为 `(out_channels, in_channels, kernel_size, kernel_size)`。
3. 为每个输出通道创建一个偏置，形状为 `(out_channels,)`。
4. 创建 `cache`，用于保存前向传播中反向传播需要的数据。

#### 阶段二：实现前向传播

1. 读取批量大小、输入通道数和图像高宽，并检查输入通道。
2. 调用 `zero_padding_batch_manual` 完成批量 Padding。
3. 根据输入尺寸、卷积核、Padding 和 Stride 计算输出高宽。
4. 创建形状为 `(batch_size, out_channels, output_h, output_w)` 的输出数组。
5. 使用两层循环遍历输出空间位置，再遍历所有输出通道。
6. 截取局部区域，将其与当前输出通道的卷积核逐元素相乘。
7. 对输入通道和卷积核空间维度求和，加入偏置并写入输出。
8. 保存输入形状和填充后的输入，最后返回输出。

#### 阶段三：实现反向传播

1. 从 `cache` 中读取输入形状和填充后的输入。
2. 创建 `grad_x_padded`、`grad_weight` 和 `grad_bias`。
3. 对 `grad_output` 在批量和空间维度求和，得到偏置梯度。
4. 按照前向传播相同的位置和输出通道进行循环。
5. 使用当前位置的上游梯度与局部输入计算卷积核梯度，并在批量维度累加。
6. 使用上游梯度与卷积核计算输入局部区域的梯度，并累加到 `grad_x_padded`。
7. 去掉 Padding 区域，得到与原输入形状相同的 `grad_x`。
8. 返回 `grad_x, grad_weight, grad_bias`。

In [ ]:
class Conv2DManual:
    """使用 NumPy 手写的多通道二维卷积层。"""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, seed=42):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # TODO 1：使用 seed 创建 NumPy 随机数生成器 rng。
        # TODO 2：计算 He 初始化比例 sqrt(2 / (in_channels * kernel_size^2))。
        # TODO 3：创建卷积核 self.weight，形状为
        #         (out_channels, in_channels, kernel_size, kernel_size)。
        # TODO 4：创建全零偏置 self.bias，形状为 (out_channels,)。
        # TODO 5：将 self.cache 初始化为 None。
        self.weight = None
        self.bias = None
        self.cache = None

    def forward(self, x):
        batch_size, channels, height, width = x.shape
        if channels != self.in_channels:
            raise ValueError("输入通道数与卷积层设置不一致")

        # TODO 6：设置 p、k、s，分别表示 Padding、卷积核大小和 Stride。
        # TODO 7：调用 zero_padding_batch_manual 得到 x_padded。
        # TODO 8：计算 output_h 和 output_w。
        # TODO 9：创建正确形状的全零输出数组 output。
        # TODO 10：遍历 out_y、out_x 和 out_channel。
        # TODO 11：根据 Stride 计算局部区域起点，并从 x_padded 截取 patch。
        # TODO 12：将 patch 与当前输出通道的卷积核逐元素相乘，
        #          在输入通道和卷积核空间维度求和，再加入偏置。
        # TODO 13：保存 (x.shape, x_padded) 到 self.cache，并返回 output。
        raise NotImplementedError("请按照 TODO 6～13 补全卷积前向传播")

    def backward(self, grad_output):
        # TODO 14：从 self.cache 读取 input_shape 和 x_padded。
        # TODO 15：设置 p、k、s，并读取 grad_output 的输出高宽。
        # TODO 16：创建 grad_x_padded、grad_weight 和 grad_bias。
        # TODO 17：遍历 out_y、out_x 和 out_channel，并截取对应 patch。
        # TODO 18：取得 local_grad，并扩展为 (N, 1, 1, 1) 形状。
        # TODO 19：累加卷积核梯度：expanded_grad * patch，并在批量维度求和。
        # TODO 20：累加输入梯度：expanded_grad * 当前输出通道的卷积核。
        # TODO 21：当 p > 0 时去掉 Padding；否则直接使用 grad_x_padded。
        # TODO 22：检查 grad_x 与原输入形状一致，并返回三个梯度。
        raise NotImplementedError("请按照 TODO 14～22 补全卷积反向传播")

## 6. 增加池化层与分类损失

基础卷积只能得到特征图。为了完成分类训练，本节继续增加：

- **最大池化**：前向传播保留每个局部区域的最大值；反向传播只将梯度传回最大值所在位置。
- **Softmax**：将全连接层输出转换为类别概率。
- **交叉熵损失**：度量预测概率与真实标签之间的差异，并提供反向传播的起点。

In [ ]:
class MaxPool2DManual:
    """手写无重叠最大池化层。"""

    def __init__(self, pool_size=2):
        self.pool_size = pool_size
        self.cache = None

    def forward(self, x):
        batch_size, channels, height, width = x.shape
        p = self.pool_size
        output_h = height // p
        output_w = width // p
        output = np.zeros((batch_size, channels, output_h, output_w))
        max_indices = np.zeros((batch_size, channels, output_h, output_w), dtype=np.int64)

        for out_y in range(output_h):
            for out_x in range(output_w):
                patch = x[:, :, out_y * p:(out_y + 1) * p, out_x * p:(out_x + 1) * p]
                flat_patch = patch.reshape(batch_size, channels, -1)
                max_indices[:, :, out_y, out_x] = np.argmax(flat_patch, axis=2)
                output[:, :, out_y, out_x] = np.max(flat_patch, axis=2)

        self.cache = (x.shape, max_indices)
        return output

    def backward(self, grad_output):
        input_shape, max_indices = self.cache
        batch_size, channels, output_h, output_w = grad_output.shape
        p = self.pool_size
        grad_x = np.zeros(input_shape)

        for out_y in range(output_h):
            for out_x in range(output_w):
                indices = max_indices[:, :, out_y, out_x]
                for position in range(p * p):
                    row = position // p
                    col = position % p
                    mask = indices == position
                    grad_x[:, :, out_y * p + row, out_x * p + col] += (
                        grad_output[:, :, out_y, out_x] * mask
                    )

        return grad_x


def softmax_cross_entropy(logits, labels):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    batch_size = logits.shape[0]
    loss = -np.mean(np.log(probabilities[np.arange(batch_size), labels] + 1e-12))

    grad_logits = probabilities.copy()
    grad_logits[np.arange(batch_size), labels] -= 1.0
    grad_logits /= batch_size
    return loss, probabilities, grad_logits

## 7. 组装两层卷积网络并实现反向传播

网络前向传播依次执行卷积、激活、池化、展平和分类。为了训练参数，前向传播必须保存反向计算需要的中间结果。

反向传播按照与前向传播相反的顺序进行：

```text
损失 → 全连接层 → Pool2 → ReLU2 → Conv2 → Pool1 → ReLU1 → Conv1
```

得到梯度后，使用 `参数 = 参数 - 学习率 × 梯度` 更新所有卷积核、卷积偏置、全连接权重和全连接偏置。

In [ ]:
class MultiLayerCNNManual:
    def __init__(self, num_classes=10, seed=42):
        self.conv1 = Conv2DManual(1, 4, kernel_size=3, padding=1, seed=seed)
        self.pool1 = MaxPool2DManual(pool_size=2)
        self.conv2 = Conv2DManual(4, 8, kernel_size=3, padding=1, seed=seed + 1)
        self.pool2 = MaxPool2DManual(pool_size=2)

        rng = np.random.default_rng(seed + 2)
        flattened_size = 8 * 2 * 2
        self.fc_weight = rng.normal(
            0.0,
            np.sqrt(2.0 / flattened_size),
            size=(flattened_size, num_classes),
        )
        self.fc_bias = np.zeros(num_classes)
        self.forward_cache = None

    def forward(self, x):
        conv1_output = self.conv1.forward(x)
        relu1_output = relu_manual(conv1_output)
        pool1_output = self.pool1.forward(relu1_output)

        conv2_output = self.conv2.forward(pool1_output)
        relu2_output = relu_manual(conv2_output)
        pool2_output = self.pool2.forward(relu2_output)

        flattened = pool2_output.reshape(x.shape[0], -1)
        logits = flattened @ self.fc_weight + self.fc_bias

        self.forward_cache = (
            conv1_output,
            pool1_output,
            conv2_output,
            pool2_output.shape,
            flattened,
        )
        return logits

    def train_batch(self, x, labels, learning_rate):
        logits = self.forward(x)
        loss, probabilities, grad_logits = softmax_cross_entropy(logits, labels)

        conv1_output, pool1_output, conv2_output, pool2_shape, flattened = self.forward_cache

        grad_fc_weight = flattened.T @ grad_logits
        grad_fc_bias = grad_logits.sum(axis=0)
        grad_flattened = grad_logits @ self.fc_weight.T

        grad_pool2 = grad_flattened.reshape(pool2_shape)
        grad_relu2 = self.pool2.backward(grad_pool2)
        grad_conv2_output = relu_backward_manual(grad_relu2, conv2_output)
        grad_pool1, grad_conv2_weight, grad_conv2_bias = self.conv2.backward(grad_conv2_output)

        grad_relu1 = self.pool1.backward(grad_pool1)
        grad_conv1_output = relu_backward_manual(grad_relu1, conv1_output)
        _, grad_conv1_weight, grad_conv1_bias = self.conv1.backward(grad_conv1_output)

        self.fc_weight -= learning_rate * grad_fc_weight
        self.fc_bias -= learning_rate * grad_fc_bias
        self.conv2.weight -= learning_rate * grad_conv2_weight
        self.conv2.bias -= learning_rate * grad_conv2_bias
        self.conv1.weight -= learning_rate * grad_conv1_weight
        self.conv1.bias -= learning_rate * grad_conv1_bias

        predictions = np.argmax(probabilities, axis=1)
        accuracy = np.mean(predictions == labels)
        return loss, accuracy

    def predict(self, x, batch_size=128):
        predictions = []
        for start in range(0, len(x), batch_size):
            logits = self.forward(x[start:start + batch_size])
            predictions.append(np.argmax(logits, axis=1))
        return np.concatenate(predictions)

    def accuracy(self, x, labels):
        return np.mean(self.predict(x) == labels)

### 7.1 检查各层输出形状

In [ ]:
shape_model = MultiLayerCNNManual(seed=RANDOM_STATE)
sample_batch = X_train[:4]

z1 = shape_model.conv1.forward(sample_batch)
a1 = relu_manual(z1)
p1 = shape_model.pool1.forward(a1)
z2 = shape_model.conv2.forward(p1)
a2 = relu_manual(z2)
p2 = shape_model.pool2.forward(a2)

print("input :", sample_batch.shape)
print("conv1 :", z1.shape)
print("pool1 :", p1.shape)
print("conv2 :", z2.shape)
print("pool2 :", p2.shape)
print("flatten:", p2.reshape(len(sample_batch), -1).shape)

## 8. 增加小批量训练循环

完整训练循环需要重复执行以下步骤：

1. 随机打乱训练样本。
2. 取出一个小批量图像与标签。
3. 前向传播得到类别得分。
4. 计算 Softmax 交叉熵和当前批次准确率。
5. 反向传播得到所有参数的梯度。
6. 使用梯度下降更新参数。
7. 每轮结束后在测试集上评价泛化效果。

可以修改 `EPOCHS`、`BATCH_SIZE` 和 `LEARNING_RATE` 观察不同超参数的影响。

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.06

model = MultiLayerCNNManual(num_classes=10, seed=RANDOM_STATE)
history = {"loss": [], "train_accuracy": [], "test_accuracy": []}
rng = np.random.default_rng(RANDOM_STATE)

for epoch in range(1, EPOCHS + 1):
    indices = rng.permutation(len(X_train))
    epoch_loss = 0.0
    epoch_correct = 0.0
    sample_count = 0

    for start in range(0, len(X_train), BATCH_SIZE):
        batch_indices = indices[start:start + BATCH_SIZE]
        batch_x = X_train[batch_indices]
        batch_y = y_train[batch_indices]

        batch_loss, batch_accuracy = model.train_batch(batch_x, batch_y, LEARNING_RATE)
        epoch_loss += batch_loss * len(batch_x)
        epoch_correct += batch_accuracy * len(batch_x)
        sample_count += len(batch_x)

    average_loss = epoch_loss / sample_count
    train_accuracy = epoch_correct / sample_count
    test_accuracy = model.accuracy(X_test, y_test)

    history["loss"].append(average_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_accuracy"].append(test_accuracy)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"loss={average_loss:.4f} | "
        f"train_acc={train_accuracy:.4f} | "
        f"test_acc={test_accuracy:.4f}"
    )

## 9. 可视化训练过程

In [ ]:
epochs_axis = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_axis, history["loss"], marker="o", markersize=3)
axes[0].set_title("training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_axis, history["train_accuracy"], label="training set")
axes[1].plot(epochs_axis, history["test_accuracy"], label="test set")
axes[1].set_title("classification accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1.02)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. 测试结果与混淆矩阵

In [ ]:
test_predictions = model.predict(X_test)
test_accuracy = np.mean(test_predictions == y_test)
matrix = confusion_matrix(y_test, test_predictions)

print(f"final test accuracy: {test_accuracy:.2%}")

fig, ax = plt.subplots(figsize=(7, 6))
image_handle = ax.imshow(matrix, cmap="Blues")
fig.colorbar(image_handle, ax=ax)
ax.set_title("test set confusion matrix")
ax.set_xlabel("predicted labels")
ax.set_ylabel("true labels")
ax.set_xticks(range(10))
ax.set_yticks(range(10))

threshold = matrix.max() / 2
for row in range(10):
    for col in range(10):
        color = "white" if matrix[row, col] > threshold else "black"
        ax.text(col, row, matrix[row, col], ha="center", va="center", color=color)

plt.tight_layout()
plt.show()

## 11. 可视化预测样本

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
for index, ax in enumerate(axes.ravel()):
    true_label = y_test[index]
    predicted_label = test_predictions[index]
    title_color = "green" if true_label == predicted_label else "red"

    ax.imshow(X_test[index, 0], cmap="gray")
    ax.set_title(f"true:{true_label} predicted:{predicted_label}", color=title_color)
    ax.axis("off")

plt.suptitle("green: correct, red: incorrect")
plt.tight_layout()
plt.show()

## 12. 可视化两层卷积特征图

第一层特征图直接响应原始图像中的局部边缘；第二层特征图融合第一层多个通道的信息，通常更抽象。

In [ ]:
sample = X_test[0:1]
feature1 = relu_manual(model.conv1.forward(sample))
pooled1 = model.pool1.forward(feature1)
feature2 = relu_manual(model.conv2.forward(pooled1))

fig, axes = plt.subplots(2, 8, figsize=(15, 5))

for index in range(8):
    if index < feature1.shape[1]:
        axes[0, index].imshow(feature1[0, index], cmap="viridis")
        axes[0, index].set_title(f"Conv1-{index + 1}")
    else:
        axes[0, index].axis("off")

    axes[1, index].imshow(feature2[0, index], cmap="viridis")
    axes[1, index].set_title(f"Conv2-{index + 1}")

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(f"feature maps of test digit {y_test[0]}")
plt.tight_layout()
plt.show()

## 13. 实验练习

1. 将第一层和第二层的卷积核数量分别改为 6 和 12，观察准确率和速度。
2. 去掉第二个卷积层，与多层网络的训练结果进行比较。
3. 修改卷积层的 padding，并同步调整全连接层输入尺寸。
4. 调整学习率、批量大小和训练轮数，观察训练曲线变化。
5. 找出测试集中预测错误的样本，分析容易混淆的数字类别。

### 实验总结

本实验从底层算子出发，手写实现了两层卷积网络的前向传播和反向传播。两层卷积先后提取低层和高层特征，再通过全连接层完成数字分类。